In [1]:
!pip install vectorbt

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime
import pytz
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import vectorbt as vbt

In [3]:
# vbt.settings.array_wrapper['freq'] = 'days'
# vbt.settings.returns['year_freq'] = '252 days'
# vbt.settings.portfolio['seed'] = 42
# vbt.settings.portfolio.stats['incl_unrealized'] = True

all_symbols = ['VTSAX', 'VBMFX', 'VGTSX', 'VTABX']
benchmark_symbol = 'VFIFX'
CASH_SYMBOL = 'VBMFX'

# Дата появления VTABX
VTABX_START_DATE = datetime(2013, 6, 4, tzinfo=pytz.utc)
# Дата когда VTABX накапливает достаточно данных для использования (через год)
VTABX_ACTIVE_DATE = datetime(2014, 6, 4, tzinfo=pytz.utc)

start_date_benchmark = datetime(2008, 1, 1, tzinfo=pytz.utc)
end_date_benchmark = datetime(2024, 12, 31, tzinfo=pytz.utc)

start_date_symbols = datetime(2007, 1, 1, tzinfo=pytz.utc)
end_date_symbols = datetime(2024, 12, 31, tzinfo=pytz.utc)

composition_data = {
    'Year':  [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025],
    'VTSAX': [71.6, 71.6, 71.5, 62.9, 62.6, 63.1, 63.1, 62.9, 54.2, 53.9, 53.8, 54.2, 53.9, 54.2, 54.7, 54.1, 54.3, 52.6],
    'VBMFX': [10.0,  9.9, 10.1, 10.1, 10.0,  9.9,  8.1,  8.1,  7.1,  7.1,  6.9,  7.1, 6.4,  6.6,  6.8,  6.7,  7.0,  7.0],
    'VGTSX': [18.2, 18.1, 17.9, 26.9, 27.1, 27.0, 26.9, 27.3, 35.6, 36.1, 36.2, 36.0, 36.0, 36.4, 35.7, 36.3, 36.0, 36.8],
    'VTABX': [0.0,  0.0,  0.0,  0.0,  0.0,  0.0,  2.0,  3.0,  2.9,  3.0,  3.9,  3.0,  3.0,  3.0,  2.8,  3.0,  3.1,  3.1],
}

# Инициализируем symbols как все возможные символы для загрузки данных
symbols = all_symbols

print(f"VTABX start date: {VTABX_START_DATE.date()}")
print(f"VTABX active date: {VTABX_ACTIVE_DATE.date()}")
print(f"All symbols for data loading: {symbols}")

# Параметры для бектестинга
initial_cash = 10000.0
transaction_cost = 0.002

# crisis_labels = pd.read_csv('/Users/lina4688/VS Code Projects/vectorbt_v2_25_09_28/selected_prices.csv', index_col=0, parse_dates=True)
# crisis_labels = crisis_labels['risk_label_smoothed']
# print("selected_prices_df loaded. Shape:", crisis_labels)


VTABX start date: 2013-06-04
VTABX active date: 2014-06-04
All symbols for data loading: ['VTSAX', 'VBMFX', 'VGTSX', 'VTABX']


In [4]:
# Загрузка данных с обработкой отсутствующих данных для VTABX
print(f"Benchmark period: {start_date_benchmark.date()} to {end_date_benchmark.date()}")
print(f"Stock symbols period: {start_date_symbols.date()} to {end_date_symbols.date()}")

# Загружаем данные для всех символов
benchmark_data = vbt.YFData.download(benchmark_symbol, start=start_date_benchmark, end=end_date_benchmark)
stocks_data = vbt.YFData.download(symbols, start=start_date_symbols, end=end_date_symbols)
risk_free_data = yf.download('^IRX', start=start_date_symbols, end=end_date_symbols)

# Save the downloaded data with dates in the filename
benchmark_fname = f"benchmark_data_{start_date_benchmark.date()}_{end_date_benchmark.date()}.pkl"
stocks_fname = f"stocks_data_{start_date_symbols.date()}_{end_date_symbols.date()}.pkl"
risk_free_fname = f"risk_free_data_{start_date_symbols.date()}_{end_date_symbols.date()}.pkl"

benchmark_data.save(benchmark_fname)
stocks_data.save(stocks_fname)
risk_free_data.to_pickle(risk_free_fname)
print(f"Saved benchmark_data to {benchmark_fname}")
print(f"Saved stocks_data to {stocks_fname}")
print(f"Saved risk_free_data to {risk_free_fname}")

# print(f"Benchmark data shape: {benchmark_price.shape}")
# print(f"Stocks data shape: {stocks_price.shape}")
# print(f"Benchmark date range: {benchmark_price.index[0].date()} to {benchmark_price.index[-1].date()}")
# print(f"Stocks date range: {stocks_price.index[0].date()} to {stocks_price.index[-1].date()}\n")

# # Проверяем доступность данных для VTABX
# vtabx_first_valid = stocks_price['VTABX'].first_valid_index()
# print(f"VTABX first valid date: {vtabx_first_valid}")

# # --- Risk free rate ---
# risk_free_data = yf.download('^IRX', start=start_date_symbols, end=end_date_symbols)
# risk_free_rate = risk_free_data['Close'] / 100

# # Определяем price для дальнейшего использования
# price = stocks_price

# # Исправляем проблему с временными зонами
# # Приводим оба индекса к одному формату (tz-naive)
# price.index = pd.to_datetime(price.index.date)
# risk_free_rate.index = pd.to_datetime(risk_free_rate.index.date)

# # Обрезаем selected_prices_df под stock symbols period
# crisis_labels = crisis_labels.loc[
#     (crisis_labels.index >= price.index[0]) & (crisis_labels.index <= price.index[-1])
# ]
# print("selected_prices_df clipped to stock symbols period. Shape:", crisis_labels.shape)

# # Fill missing days in risk_free_rate using previous available value
# missing_days = price.index.difference(risk_free_rate.index)
# print("Дней не хватает в risk_free_rate:", len(missing_days))

# for missing_date in missing_days:
#     prev_date = risk_free_rate.index[risk_free_rate.index < missing_date].max()
#     if pd.isna(prev_date):
#         raise ValueError(f"Нет предыдущего значения для вставки в risk_free_rate для {missing_date.date()}")
#     prev_value = risk_free_rate.loc[prev_date]
#     risk_free_rate.loc[missing_date] = prev_value
# risk_free_rate = risk_free_rate.sort_index()

# print("Any NaNs in risk_free_rate?", risk_free_rate.isna().any().any())

# trading_start = start_date_benchmark
# trading_end = end_date_benchmark
# print(f"Trading period: {trading_start.date()} to {trading_end.date()}")

# normalized_prices = price / price.iloc[0]
# returns = price.pct_change()
# ann_factor = returns.vbt.returns.ann_factor

# print(f"Price data shape: {price.shape}")
# print(f"Available symbols: {list(price.columns)}")
# print(f"Date range: {price.index[0].date()} to {price.index[-1].date()}")


Benchmark period: 2008-01-01 to 2024-12-31
Stock symbols period: 2007-01-01 to 2024-12-31


/home/duser/.local/lib/python3.12/site-packages/vectorbt/data/base.py:527: UserWarning:

Symbols have mismatching index. Setting missing data points to NaN.

/tmp/ipykernel_2976418/2700402134.py:8: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed

Saved benchmark_data to benchmark_data_2008-01-01_2024-12-31.pkl
Saved stocks_data to stocks_data_2007-01-01_2024-12-31.pkl
Saved risk_free_data to risk_free_data_2007-01-01_2024-12-31.pkl
